# 04 - Flower Runtime Orchestration

Rendered snapshot of how the deployment runtime is actuated in this demo. Captured at `2026-05-14T16:09:13` from the local Docker/Flower topology.

The goal of this notebook is not to retrain the model. It shows which SuperLinks, SuperNodes and SuperExec services are running, what each one connects to, and which `flwr run` calls drive the regional and global layers.

## 1. Running Services

There are three Flower federations running at the same time: `region-eu`, `region-na`, and `global`. Healthcare-site SuperNodes connect to regional SuperLinks. Region gateway SuperNodes connect only to the global SuperLink.

In [1]:
!docker compose ps --format 'table {{.Service}}\t{{.State}}\t{{.Ports}}'

{.Service}   {.State}   {.Ports}


## 2. Compose Roles

`SuperLink` services expose the Flower APIs. `SuperExec ServerApp` services execute the server application submitted through `flwr run`. Healthcare-site `SuperNode` services receive tasks from their regional SuperLink, while gateway SuperNodes receive tasks from the global SuperLink. ClientApp SuperExec services execute `hfl_cicids.client_app` next to each SuperNode.

In [2]:
from pathlib import Path
import pandas as pd
import yaml

def service_role(name):
    if name.endswith('-superlink'):
        return 'SuperLink'
    if name.endswith('-superexec-serverapp'):
        return 'SuperExec ServerApp'
    if name.endswith('-supernode'):
        return 'SuperNode'
    if name.endswith('-superexec-clientapp'):
        return 'SuperExec ClientApp'
    return 'service'

def list_arg(args, flag):
    return args[args.index(flag) + 1] if flag in args else ''

compose = yaml.safe_load(Path('docker-compose.yml').read_text())
rows = []
for name, service in sorted(compose['services'].items()):
    if name.startswith('flat-'):
        continue
    command = service.get('command', [])
    rows.append({
        'service': name,
        'role': service_role(name),
        'superlink_target': list_arg(command, '--superlink'),
        'appio_target': list_arg(command, '--appio-api-address'),
        'node_config': list_arg(command, '--node-config'),
        'volumes': ', '.join(service.get('volumes', [])),
    })
pd.DataFrame(rows)

,service,role,superlink_target,appio_target,node_config,volumes
0,global-superexec-serverapp,SuperExec ServerApp,,global-superlink:9091,,./shared:/shared
1,global-superlink,SuperLink,,,,
2,hospital-eu-01-superexec-clientapp,SuperExec ClientApp,,hospital-eu-01-supernode:9094,,"./data/partitions/hospital_eu_01:/data:ro, ./s..."
3,hospital-eu-01-supernode,SuperNode,region-eu-superlink:9092,,"role=""hospital"" region=""region_eu"" hospital-id...",
4,hospital-eu-02-superexec-clientapp,SuperExec ClientApp,,hospital-eu-02-supernode:9094,,"./data/partitions/hospital_eu_02:/data:ro, ./s..."
5,hospital-eu-02-supernode,SuperNode,region-eu-superlink:9092,,"role=""hospital"" region=""region_eu"" hospital-id...",
6,hospital-eu-03-superexec-clientapp,SuperExec ClientApp,,hospital-eu-03-supernode:9094,,"./data/partitions/hospital_eu_03:/data:ro, ./s..."
7,hospital-eu-03-supernode,SuperNode,region-eu-superlink:9092,,"role=""hospital"" region=""region_eu"" hospital-id...",
8,hospital-na-01-superexec-clientapp,SuperExec ClientApp,,hospital-na-01-supernode:9094,,"./data/partitions/hospital_na_01:/data:ro, ./s..."
9,hospital-na-01-supernode,SuperNode,region-na-superlink:9092,,"role=""hospital"" region=""region_na"" hospital-id...",


## 3. Flower Profiles

The profile names are what the orchestrator passes to `flwr run . <profile>`. Each profile points to one SuperLink control endpoint.

In [3]:
!flwr config list

Flower Config file: /Users/david/.flwr/config.toml
SuperLink connections:
  region-eu (default)
  region-na
  global
  flat


## 4. Actuating the Federations

`scripts/run_hierarchical_rounds.py` is the control plane for the demo. For each global round it submits one regional Flower run per region, then one global Flower run where the clients are the gateway SuperNodes.

In [4]:
!python scripts/run_hierarchical_rounds.py --global-rounds 1 --regional-rounds 1 --batch-size 8192 --dry-run

────────────────────────────────────────────────────────────────────────────────────────────────────── Global round 1 ──────────────────────────────────────────────────────────────────────────────────────────────────────
Starting from global checkpoint: shared/checkpoints/global/round_0.pt

Regional phase: region_eu | site_nodes=3 | train_examples=871,173 | regional_rounds=1

Submitting Flower run to region-eu
$ flwr run . region-eu --stream --run-config 'level="regional" region="region_eu" global-round=1 init-checkpoint="/shared/checkpoints/global/round_0.pt" output-checkpoint="/shared/checkpoints/region_eu/round_1.pt" num-server-rounds=1 local-epochs=1 batch-size=8192 learning-rate=0.001 input-dim=78 region-num-examples=871173 fraction-train=1.0 fraction-evaluate=1.0 min-train-nodes=3 min-evaluate-nodes=3 min-available-nodes=3'

Regional phase: region_na | site_nodes=3 | train_examples=1,110,344 | regional_rounds=1

Submitting Flower run to region-na
$ flwr run . region-na --stream -

## 5. Runtime Logs

The healthcare-site SuperNode receives train/evaluate messages from its regional SuperLink. The gateway SuperNode receives a train message from the global SuperLink and returns the regional checkpoint as a model update.

In [5]:
!docker compose logs --tail=12 hospital-eu-01-supernode region-eu-gateway-supernode region-eu-superlink global-superlink

## 6. Checkpoint Metadata

Regional checkpoints carry the regional training-example count used by global weighted FedAvg. The global checkpoint is produced by the global Flower federation.

In [6]:
from pathlib import Path
import json

for path in sorted(Path('shared/checkpoints').rglob('*.metadata.json')):
    print(path)
    print(json.dumps(json.loads(path.read_text()), indent=2, sort_keys=True))

shared/checkpoints/global/round_0.metadata.json
{
  "global_round": 0,
  "initialized": true,
  "input_dim": 78,
  "level": "global"
}
shared/checkpoints/global/round_1.metadata.json
{
  "global_round": 1,
  "input_dim": 78,
  "level": "global",
  "num_examples": 1981517,
  "weighted_by_key": "num-examples"
}
shared/checkpoints/region_eu/round_1.metadata.json
{
  "final_evaluate_metrics": {
    "eval_acc": 0.912716948789372,
    "eval_auprc": 0.7482221932009319,
    "eval_balanced_acc": 0.9465330678331263,
    "eval_f1": 0.24754852964520757,
    "eval_false_negative_rate": 0.01696630299632639,
    "eval_false_positive_rate": 0.08996756133742079,
    "eval_precision": 0.15649223825354358,
    "eval_recall": 0.9830336970036735,
    "eval_roc_auc": 0.9891377786878964
  },
  "final_train_metrics": {
    "train_loss": 0.45856640012897376,
    "val_auprc": 0.7455749061804295,
    "val_f1": 0.2770287525601163,
    "val_false_negative_rate": 0.019014734572795518,
    "val_false_positive_rate":

## 7. Evaluation Snapshot

These metrics are from the latest rendered local run. They are evidence that the global checkpoint can be loaded and evaluated across the site test splits.

In [7]:
import pandas as pd
pd.read_csv('reports/metrics_summary_global.csv')

,run,model_kind,scope,weighted_f1,macro_f1,weighted_roc_auc,weighted_auprc,mean_false_positive_rate,worst_false_negative_rate,num_examples
0,hierarchical_fl,hierarchical_global,global_summary,0.65235,0.797289,0.982793,0.8152,0.047693,0.242705,424614


## 8. Privacy Boundary Check

`shared/` is allowed to contain checkpoints, metrics and preprocessing metadata. It should not contain site CSV/parquet network-flow rows.

In [8]:
from pathlib import Path
sorted(list(Path('shared').rglob('*.csv')) + list(Path('shared').rglob('*.parquet')))

[]